# Automatisation des balises

Les documents XML que l'on passe par ce script doivent être formaté pour que chaque paragraphe soit sur une seule ligne. Ainsi les noms ne sont pas coupés par des retours à la ligne et des espaces qui pourraient empêcher la reconnaissance de certains noms lors de l'annotation automatique. 

## Chargement des bibliothèques

In [3]:
from bs4 import BeautifulSoup, NavigableString
import re
import csv

## Création des fonctions

### Lire les fichiers csv et txt

Pour les 3 fonctions qui suivent on va lire les données que l'on veut annoter dans des documents csv et txt. 
Ils sont déjà existant dans le dossier `fichier_noms`. 

#### Fonction pour lire les noms d'auteurs à partir d'un fichier .csv

Le fichier `auteurs.csv` contient les noms de l'IndexPersonne que l'on peut annoter ou tenter d'annoter automatiquement.

Les colonnes sont nommées ainsi : `xml:id,Nom,NomITA,`


In [22]:
def lire_noms_indexPers(fichier_noms): #Définition d'une fonction pour lire les fichiers individuels avec les noms et les identifiants. En créer un dictionnaire avec une clé et une valeur. 
    noms_auteurs = [] #On créer une liste vide qui va contenir le dictionnaire.
    with open(fichier_noms, 'r', encoding='utf-8') as f: #ouvre le fichier qui sera renseigné en tant que fichier_noms
        reader = csv.DictReader(f)  # Utilise DictReader pour lire les noms de colonne
        for row in reader: #Pour chaque ligne dans les colonnes de reader qui est défini au dessus, on execute les manips qui suivent.
            nom = row['Noms'].strip()  # Récupérer la colonne 'Nom'
            id_noms = row['ID'].strip()  # Récupérer la colonne 'Id'
            if nom and id_noms: # Si on a bien un nom et un id on ajoute au dico (avec .append)
                noms = [n.strip() for n in nom.split(',') if n.strip()]
                noms_auteurs.append({'xml:id': id_noms, 'Nom': noms})  # Ajouter au dictionnaire avec 'id' et 'Nom'
    return noms_auteurs


#### Fonction pour lire les noms de lieu à partir d'un fichier .csv

Le fichier `lieux.csv` contient les lieux de l'IndexLieux que l'on peut annoter ou tenter d'annoter automatiquement.

Les colonnes sont nommées ainsi : `Continent,Pays,Nom,Id,NomIta,,`

Lorsqu'il y plusieurs possibilité pour un seul xml:id, ils sont tous dans la même "case" et seront prit en compte dans la liste comme : `{'id': 'Toulouse', 'Nom': ['Toulouse', 'Tolose']}`.

In [1]:
def lire_noms_indexLieux(fichier_lieux):
    noms_lieux = []
    with open(fichier_lieux, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)  # Utilise DictReader pour lire les noms de colonne
        for row in reader:
            lieu = row['Noms'].strip()  # Récupérer la colonne 'Nom'
            id_lieu = row['ID'].strip()  # Récupérer la colonne 'Id'
            if lieu and id_lieu:
                lieux = [n.strip() for n in lieu.split(',') if n.strip()] # permet de récupérer tous les noms d'un lieu qui sont dans une seule case, séparés par une ","
                noms_lieux.append({'id': id_lieu, 'Nom': lieux})  # Ajouter au dictionnaire avec 'id' et 'Noms'
    
    return noms_lieux

Cette cellule ne sert qu'à vérifier la liste des lieux créée par la fonction lire_noms_lieux()

In [2]:
fichier_lieux = 'fichiers_noms/lieux.csv'
lire_noms_indexLieux(fichier_lieux)

NameError: name 'csv' is not defined

### Fonction qui ajoute les balises `<persName>` et `<placeName>`

Cette fonction imite la fonctionnement de la fonction Rechercher/Remplacer en utilisant des expressions régulières.

/!\ Si le document contient déjà des balises, elles ne seront pas prises en compte. On peut alors se retrouver avec un document contentant des balises doublées. 
    De la même façon si on fait tourner un fichier qui a déjà été passé par ce notebook toutes les balises vont se dédoubler.

In [25]:
def ajouter_balise(texte, noms_lieux, noms_auteurs): #Définition de la fonction "rechercher" pour les mots qui sont dans les différents dictionnaires qui sont créer au dessus.
    
    # Remplacer chaque nom d'auteur par une balise <persName>
    for auteur in noms_auteurs: #Pour les auteurs (valeur du dictionnaire noms_auteurs)
        for pers in auteur['Nom'] :
            texte = re.sub(rf"(?<![#><\w]){pers}(?!<)\b", f'<persName ref="#{auteur["xml:id"]}">{pers}</persName>', texte)
        # re = utilisation des expressions régulières. 
        # .sub permet de remplacer avec les attribut sous cette forme là (valeur recherchée, remplacement, document cible)
        # (?<![#><\w]) Regarde la non présence des caractères entre crochet avant pers. (pour empêcher le balisage des noms dans les ref)
    
    # Remplacer chaque lieu par une balise <placeName>
    for lieu in noms_lieux:
        for nom in lieu['Nom']: # test pour toutes les possibilités de Nom dans la variable nom
            texte = re.sub(rf"(?<![#><\w]){nom}(?!<)\b", f'<placeName ref="#{lieu["id"]}">{nom}</placeName>', texte)

    
    return texte

#### Fonction pour ajouter les balises `<dates>`

In [26]:
def lire_date(texte, pattern):
    # Utiliser re.sub avec une fonction de remplacement
    def replacer(match):
        year = match.group()  # Extraire la correspondance
        return f'<date when="{year}">{year}</date>'
    
    # Appliquer re.sub pour remplacer toutes les correspondances
    texte = re.sub(pattern, replacer, texte)
    return texte

#### Manipuler du XML avec BeautifulSoup

Fonction qui utilise les autres fonctions créées ci-dessus, en lisant les fichiers d'entrés et en rajoutant les balises.  
La fonction remplace les caractères qui posent soucis (les chevrons).  
Enfin elle ouvre un fichier de sortie xml.  

In [27]:
def ajouter_balises_xml(fichier_xml, fichier_noms, fichier_lieux):
    # Lire les noms d'auteurs à partir du fichier. utilise la fonction que l'on a crée au dessus pour créer les dictionnaires à partir des docs csv. On les appelle dans des variables pour les utiliser.
    noms_auteurs = lire_noms_indexPers(fichier_noms)
    noms_lieux = lire_noms_indexLieux(fichier_lieux)
    pattern = r"\b\d{4}\b" # pattern pour la reconnaissance des dates

    # Lire le fichier XML avec BeautifulSoup
    with open(fichier_xml, 'r', encoding='utf-8') as fichier:
        contenu_xml = fichier.read()

    # Utiliser BeautifulSoup pour parser le fichier XML
    soup = BeautifulSoup(contenu_xml, 'xml')

    # Parcourir les éléments du XML pour ajouter les balises
    for element in soup.find_all(text=True):
        if element.strip():  # Ignorer les éléments vides
            # Ajoute les balises <persName>, <placeName> autour des auteurs et lieux
            nouveau_texte = ajouter_balise(element, noms_lieux, noms_auteurs) #En utilisant la 2ème fonction créée au dessus. Mime la fonction rechercher/remplacer dans le texte.
            
            # Ajoute les balises <date> avec l'attribut @when autour des dates écrites en chiffre arabe
            nouveau_texte = lire_date(nouveau_texte, pattern)
            # Remplacer le texte existant par le nouveau texte avec balises
            element.replace_with(NavigableString(nouveau_texte))

    fichier_modifie = soup.prettify()
    # On remplace les caractères spéciaux en chevron que l'on souhaite avoir dans le doc XML
    fichier_corrige = fichier_modifie.replace("&lt;", "<").replace("&gt;", ">")

    # Sauvegarder le fichier XML modifié
    with open('fichier_modifie.xml', 'w', encoding='utf-8') as fichier_modifie:
        fichier_modifie.write(str(fichier_corrige)) #Ecrit un nouveau fichier XML à partir des modifications apportées.

    print("Le fichier XML a été modifié et sauvegardé sous 'fichier_modifie.xml'.")
    print(fichier_corrige)

### Utilisation

#### Chargement dans des variables des fichiers csv. 

Les fichiers dont on a besoin pour l'annotation des noms et des lieux sont déjà indiqué dans la cellule suivante avec le bon chemin d'accès vers le dossier `fichiers_noms`. 

In [28]:
# On assigne les fichiers que l'on veut utiliser aux variables qui sont utilisées dans la fonction qui créé les dictionnaires. 
# On met alors le chemin de chacun des fichiers dans la bonne variable.
fichier_noms = 'fichiers_noms/auteurs.csv'
fichier_lieux = 'fichiers_noms/lieux.csv'

### Le fichier à annoter

On remplace le nom de fichier pour y accéder dans le dossier où il se trouve, tel que : `../Nom_du_fichier.xml`


In [ ]:
#fichier_xml = 'exemple.xml'
#fichier_xml = '../postTransk/sortie.xml'
#fichier_xml = '../../Architecture/.xml'
#fichier_xml = '../../Perspective/.xml'
fichier_xml = '../../Peinture/Vignola_RegoleProspettivaPratica.xml'

### Exectuer le programme

C'est la dernière cellule à exécuter. Elle appelle la fonction `ajouter_balises_xml()`.

In [30]:
# C'est l'exécution de cette cellule qui fait tourner la fonction principale pour l'automatisation des balises
# A faire tourner en dernier, après avoir mit le bon fichier en entrée dans la cellule précédente.
# Il faut bien faire exécuter toutes les cellules avant celles-ci pour que la fonction puisse récupérer toutes les infos et fonctions dont elle a besoin pour tourner correctement.
ajouter_balises_xml(fichier_xml, fichier_noms, fichier_lieux)


C:\Users\ebondoer\AppData\Local\Temp\ipykernel_23820\273879894.py:15: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  for element in soup.find_all(text=True):


Le fichier XML a été modifié et sauvegardé sous 'fichier_modifie.xml'.
<?xml version="1.0" encoding="utf-8"?>
<TEI xmlns="http://www.tei-c.org/ns/1.0" xmlns:xi="http://www.w3.org/2001/XInclude">
 <teiHeader>
  <fileDesc>
   <titleStmt>
    <title xml:lang="ita">
     Vignola_RegoleProspettivaPratica
    </title>
   </titleStmt>
   <publicationStmt>
    <publisher>
     tranScriptorium
    </publisher>
   </publicationStmt>
   <sourceDesc>
    <bibl>
     <title xml:lang="ita">
      Vignola_RegoleProspettivaPratica
     </title>
     <idno type="Transkribus">
      9566158
     </idno>
    </bibl>
   </sourceDesc>
  </fileDesc>
  <profileDesc>
   <xi:include href="IndexPersonnes.xml" xpointer="element(/1/1)"/>
   <xi:include href="IndexLieux.xml" xpointer="element(/1/1)"/>
  </profileDesc>
 </teiHeader>
 <text>
  <body>
   <div>
    <pb n="1"/>
    <pb n="2"/>
    <pb n="3"/>
    <pb n="4"/>
   </div>
   <div>
    <pb n="5"/>
    <head>
     LE DUE REGOLE DELLA PROSPETTIVA PRATICA DI M